In [ ]:
import pandas as pd
import sqlite3
from hgnc.manager import DbManager
from hgnc.constants import (
    DATABASE_PATH,
    DOWNLOAD_FILE_PATH,
    COLUMNS
)

dbm = DbManager(DATABASE_PATH)
dbm._download_data()
df = pd.read_csv(
    DOWNLOAD_FILE_PATH,
    usecols=list(COLUMNS.keys()),
    index_col="HGNC ID",
    sep="\t",
)
df.rename(columns=COLUMNS, inplace=True)
df.index.rename("id", inplace=True)
conn = sqlite3.connect(DATABASE_PATH)

In [9]:
df

,approved_symbol,approved_name,alias_symbols,chromosome,accession_numbers,enzyme_ids
id,,,,,,
HGNC:5,A1BG,alpha-1-B glycoprotein,NaN,19q13.43,NaN,NaN
HGNC:37133,A1BG-AS1,A1BG antisense RNA 1,FLJ23569,19q13.43,BC040926,NaN
HGNC:24086,A1CF,APOBEC1 complementation factor,"ACF, ASP, ACF64, ACF65, APOBEC1CF",10q11.23,AF271790,NaN
HGNC:6,A1S9T,"symbol withdrawn, see [HGNC:12469](/data/gene-...",NaN,NaN,NaN,NaN
HGNC:7,A2M,alpha-2-macroglobulin,"FWP007, S863-7, CPAMD5",12p13.31,"BX647329, X68728, M11313",NaN
...,...,...,...,...,...,...
HGNC:25820,ZYG11B,"zyg-11 family member B, cell cycle regulator",FLJ13456,1p32.3,AB051517,NaN
HGNC:13200,ZYX,zyxin,NaN,7q34,X95735,NaN
HGNC:51695,ZYXP1,zyxin pseudogene 1,NaN,8q24.23,NaN,NaN


In [10]:
df.index = df.index.str.split(":").str[1].astype(int)
df

,approved_symbol,approved_name,alias_symbols,chromosome,accession_numbers,enzyme_ids
id,,,,,,
5,A1BG,alpha-1-B glycoprotein,NaN,19q13.43,NaN,NaN
37133,A1BG-AS1,A1BG antisense RNA 1,FLJ23569,19q13.43,BC040926,NaN
24086,A1CF,APOBEC1 complementation factor,"ACF, ASP, ACF64, ACF65, APOBEC1CF",10q11.23,AF271790,NaN
6,A1S9T,"symbol withdrawn, see [HGNC:12469](/data/gene-...",NaN,NaN,NaN,NaN
7,A2M,alpha-2-macroglobulin,"FWP007, S863-7, CPAMD5",12p13.31,"BX647329, X68728, M11313",NaN
...,...,...,...,...,...,...
25820,ZYG11B,"zyg-11 family member B, cell cycle regulator",FLJ13456,1p32.3,AB051517,NaN
13200,ZYX,zyxin,NaN,7q34,X95735,NaN
51695,ZYXP1,zyxin pseudogene 1,NaN,8q24.23,NaN,NaN


In [11]:
df["alias_symbol"] = df["alias_symbols"].str.split(", ")
df

,approved_symbol,approved_name,alias_symbols,chromosome,accession_numbers,enzyme_ids,alias_symbol
id,,,,,,,
5,A1BG,alpha-1-B glycoprotein,NaN,19q13.43,NaN,NaN,NaN
37133,A1BG-AS1,A1BG antisense RNA 1,FLJ23569,19q13.43,BC040926,NaN,[FLJ23569]
24086,A1CF,APOBEC1 complementation factor,"ACF, ASP, ACF64, ACF65, APOBEC1CF",10q11.23,AF271790,NaN,"[ACF, ASP, ACF64, ACF65, APOBEC1CF]"
6,A1S9T,"symbol withdrawn, see [HGNC:12469](/data/gene-...",NaN,NaN,NaN,NaN,NaN
7,A2M,alpha-2-macroglobulin,"FWP007, S863-7, CPAMD5",12p13.31,"BX647329, X68728, M11313",NaN,"[FWP007, S863-7, CPAMD5]"
...,...,...,...,...,...,...,...
25820,ZYG11B,"zyg-11 family member B, cell cycle regulator",FLJ13456,1p32.3,AB051517,NaN,[FLJ13456]
13200,ZYX,zyxin,NaN,7q34,X95735,NaN,NaN
51695,ZYXP1,zyxin pseudogene 1,NaN,8q24.23,NaN,NaN,NaN


In [12]:
df_alias_symbol = df.alias_symbol.explode()
df_alias_symbol

id
5                 NaN
37133        FLJ23569
24086             ACF
24086             ASP
24086           ACF64
             ...     
29027        KIAA0399
29027            ZZZ4
29027        FLJ10821
24523    DKFZP564I052
24523           ATAC1
Name: alias_symbol, Length: 72706, dtype: str

In [13]:
df_alias_symbol = df_alias_symbol.to_frame()
df_alias_symbol

,alias_symbol
id,
5,NaN
37133,FLJ23569
24086,ACF
24086,ASP
24086,ACF64
...,...
29027,KIAA0399
29027,ZZZ4
29027,FLJ10821


In [14]:
df_alias_symbol.reset_index(drop=True, inplace=True)
df_alias_symbol

,alias_symbol
0,NaN
1,FLJ23569
2,ACF
3,ASP
4,ACF64
...,...
72701,KIAA0399
72702,ZZZ4
72703,FLJ10821
72704,DKFZP564I052


In [15]:
df_alias_symbol.index.rename("id", inplace=True)
df_alias_symbol

,alias_symbol
id,
0,NaN
1,FLJ23569
2,ACF
3,ASP
4,ACF64
...,...
72701,KIAA0399
72702,ZZZ4
72703,FLJ10821


In [16]:
df_alias_symbol.to_sql("alias_symbol", conn, index_label="id", if_exists="replace")

72706

In [17]:
df_alias = df[["alias_symbols"]].copy()
df_alias.rename(columns={"alias_symbols": "symbol"}, inplace=True)
df_alias


,symbol
id,
5,NaN
37133,FLJ23569
24086,"ACF, ASP, ACF64, ACF65, APOBEC1CF"
6,NaN
7,"FWP007, S863-7, CPAMD5"
...,...
25820,FLJ13456
13200,NaN
51695,NaN


In [18]:
df_alias.to_sql("gene", conn, index_label="id", if_exists="replace")

50263

In [19]:
sql = """
    SELECT 
        g.symbol, s.synonym
    FROM 
        gene AS g INNER JOIN synonym AS s 
        ON (g.id = s.gene_id)
    WHERE s.synonym LIKE 'A%'
    GROUP BY g.symbol, s.synonym
"""

pd.read_sql(sql, conn)

DatabaseError: Execution failed on sql '
    SELECT 
        g.symbol, s.synonym
    FROM 
        gene AS g INNER JOIN synonym AS s 
        ON (g.id = s.gene_id)
    WHERE s.synonym LIKE 'A%'
    GROUP BY g.symbol, s.synonym
': no such table: synonym